### 线性回归的简洁实现

In [24]:
import numpy as np
import torch
from d2l import torch as d2l
from torch.utils import data

In [25]:
def synthetic_data(w, b, num_examples):
    """生成y = Xw + b"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

In [26]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

读取数据集

In [27]:
def load_array(data_arrays, batch_size, is_train = True):
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [28]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [29]:
next(iter(data_iter))

[tensor([[-2.4607e+00,  4.6313e-01],
         [ 5.3217e-01,  1.0945e+00],
         [ 7.6619e-01, -5.7616e-01],
         [-8.6329e-01, -1.0503e+00],
         [ 1.2495e+00, -4.2266e-01],
         [-3.5483e-01, -3.6370e-01],
         [ 1.8374e+00, -8.0623e-01],
         [ 1.0446e-03,  2.5516e-01],
         [-1.4846e-01,  2.4149e-01],
         [-2.6537e-02, -8.7205e-01]]),
 tensor([[-2.2995],
         [ 1.5321],
         [ 7.6821],
         [ 6.0435],
         [ 8.1412],
         [ 4.7301],
         [10.6230],
         [ 3.3240],
         [ 3.0925],
         [ 7.1278]])]

定义模型

In [30]:
from torch import nn
net = nn.Sequential(nn.Linear(2, 1))

初始化模型参数

In [31]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

定义损失函数：计算均方误差使用的是MSELoss类，返回所有样本损失的平均值

In [32]:
loss = nn.MSELoss()

定义优化算法，示例化一个SGD

In [33]:
trainer = torch.optim.SGD(net.parameters(), lr =  0.03)

In [34]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch:{epoch + 1}, loss:{l:f}')


epoch:1, loss:0.000323
epoch:2, loss:0.000093
epoch:3, loss:0.000093


比较生成数据集的真实参数和通过有限数据训练获得的模型参数

In [35]:
w = net[0].weight.data
print(f'w的估计误差:{true_w - w.reshape(true_w.shape)}')
b = net[0].bias.data
print(f'b的估计误差:{true_b - b}')

w的估计误差:tensor([-0.0003,  0.0003])
b的估计误差:tensor([0.0005])
